# Notebook 6: NER Extraction & Entity Frequency Analysis
---
## Strategy
A standalone **Named Entity Recognition** pipeline for Hinglish video titles.
Uses `Babelscape/wikineural-multilingual-ner` for fast multilingual entity extraction.

## Flow
```
Raw Titles -> Clean (remove hashtags, channel codes)
           -> NER Model (PER, ORG, LOC extraction)
           -> Deduplication (merge "Trump" + "Donald Trump")
           -> Frequency Map + CSV Export
```

## When to Use
- As a pre-processing step before clustering
- To understand WHO and WHERE your data talks about
- To generate the seed list for Entity-Anchored Clustering

## Output
- CSV with columns: Original Title, Persons, Organizations, Locations
- Frequency charts for top entities


In [ ]:
!pip install -q transformers torch sentencepiece protobuf pandas tqdm
print("Ready")

In [ ]:
import pandas as pd, re, html
from transformers import pipeline as hf_pipeline
from collections import Counter, defaultdict
from tqdm import tqdm
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

## Text Cleaning for NER

In [ ]:
def clean_for_ner(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"#\w+", "", text)
    channels = [r"\|?\s*ABP\s*NEWS", r"\|?\s*NDTV", r"\|?\s*AAJ\s*TAK",
               r"\|?\s*NEWS18", r"\|?\s*LIVE", r"\|?\s*Breaking"]
    for p in channels: text = re.sub(p, "", text, flags=re.I)
    text = re.sub(r"[^\w\s\d,.\-!?\u0900-\u097F]", "", text)
    text = text.strip(" |:-")
    return re.sub(r"\s+", " ", text).strip()

# --- UNIVERSAL DATA LOADER ---
import pandas as pd
import os

# Get the file extension
file_ext = os.path.splitext(filename)[1].lower()

if file_ext == '.xlsx':
    print("📂 Detected Excel file. Using read_excel...")
    df = pd.read_excel(filename)
elif file_ext == '.csv' or file_ext == '.txt':
    print("📂 Detected Text/CSV file. Using read_csv with encoding recovery...")
    try:
        df = pd.read_csv(filename, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(filename, encoding='latin1')
        except:
            df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')
else:
    print(f"⚠️ Unknown format {file_ext}. Attempting generic read...")
    df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')

# Ensure we have a title column
if "title" not in df.columns:
    if len(df.columns) == 1:
        df.columns = ["title"]
    else:
        df.columns = ["title"] + list(df.columns[1:])
# ------------------------------
if "title" not in df.columns: df.columns=["title"]+list(df.columns[1:])
titles_raw = df["title"].fillna("").astype(str).tolist()
titles_clean = [clean_for_ner(t) for t in titles_raw]
print(f"Cleaned {len(titles_clean)} titles")

## NER Extraction

In [ ]:
ner_pipe = hf_pipeline("ner", model="Babelscape/wikineural-multilingual-ner",
                       aggregation_strategy="simple")

results = []
batch_size = 32
for i in tqdm(range(0, len(titles_clean), batch_size)):
    batch = [t for t in titles_clean[i:i+batch_size] if t.strip()]
    if not batch: continue
    ner_results = ner_pipe(batch)
    for entities in ner_results:
        grouped = defaultdict(list)
        for e in entities:
            grouped[e["entity_group"]].append(e["word"].strip())
        results.append({
            "Persons": ", ".join(set(grouped.get("PER", []))),
            "Organizations": ", ".join(set(grouped.get("ORG", []))),
            "Locations": ", ".join(set(grouped.get("LOC", [])))
        })

ner_df = pd.DataFrame(results)
ner_df.insert(0, "Original Title", titles_raw[:len(ner_df)])
print(f"Extracted entities from {len(ner_df)} titles")
ner_df.head(10)

## Entity Frequency Analysis

In [ ]:
person_list = []
for p in ner_df["Persons"].dropna():
    person_list.extend([x.strip() for x in str(p).split(",") if x.strip()])

org_list = []
for o in ner_df["Organizations"].dropna():
    org_list.extend([x.strip() for x in str(o).split(",") if x.strip()])

loc_list = []
for l in ner_df["Locations"].dropna():
    loc_list.extend([x.strip() for x in str(l).split(",") if x.strip()])

print("Top 15 People:")
for n, c in Counter(person_list).most_common(15): print(f"  {n}: {c}")
print("\nTop 15 Organizations:")
for n, c in Counter(org_list).most_common(15): print(f"  {n}: {c}")
print("\nTop 15 Locations:")
for n, c in Counter(loc_list).most_common(15): print(f"  {n}: {c}")

In [ ]:
ner_df.to_csv("ner_extracted.csv", index=False)
files.download("ner_extracted.csv")